# 04 · Schema and Nested Data

> **Module 2–3 · Design an Efficient Schema**

In this notebook you will:

- Design a schema for a nested, real-world order model
- Create `RECORD`, `REPEATED`, `JSON`, and `GEOGRAPHY` columns
- Load nested data from JSON and from Parquet
- Query nested data with `UNNEST` and the JSON functions
- Evolve a schema safely with `ALTER TABLE`

---

## Setup

In [ ]:
import os
import sys
import json

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. Designing the schema

Good schema design answers three questions:

1. **What is the grain?** One row = one order? One order line?
2. **What will be queried?** Design for the query patterns, not the source shape.
3. **What will change?** Design for schema evolution.

| Decision | Choice | Why |
|----------|--------|-----|
| Grain | One row per order | Orders are queried at order grain; lines are a nested array |
| Money | `NUMERIC` | Exact decimal arithmetic — never `FLOAT64` for money |
| Timestamps | `TIMESTAMP` | Always UTC, unambiguous |
| Dates | `DATE` | Partition key, no time component |
| Status | `STRING` | Readable; a lookup table adds joins later |
| Free-form attributes | `JSON` | Schema-free for sparse, evolving data |
| Nested fixed attributes | `RECORD` | Type-safe, queryable with dot notation |

In [ ]:
ORDER_SCHEMA = [
    bigquery.SchemaField("order_id", "STRING", mode="REQUIRED", description="Unique order ID"),
    bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED", description="Customer identifier"),
    bigquery.SchemaField("order_ts", "TIMESTAMP", mode="REQUIRED", description="UTC timestamp of the order"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED", description="Business date (partition key)"),
    bigquery.SchemaField("channel", "STRING", mode="REQUIRED", description="web | mobile | store"),
    bigquery.SchemaField("region", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("currency", "STRING", mode="REQUIRED", description="ISO 4217"),
    bigquery.SchemaField("amount_local", "NUMERIC", mode="REQUIRED", description="Order total in local currency"),
    bigquery.SchemaField("amount_eur", "NUMERIC", description="Order total converted to EUR"),
    bigquery.SchemaField("status", "STRING", mode="REQUIRED", description="active | complete | cancelled | refunded"),
    bigquery.SchemaField("line_count", "INTEGER", mode="REQUIRED", description="Number of line items"),
    bigquery.SchemaField(
        name="line_items", field_type="RECORD", mode="REPEATED", description="Order lines",
        fields=[
            bigquery.SchemaField("product_id", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("quantity", "INTEGER", mode="REQUIRED"),
            bigquery.SchemaField("unit_price", "NUMERIC", mode="REQUIRED"),
            bigquery.SchemaField("discount", "NUMERIC", description="Line-level discount"),
        ],
    ),
    bigquery.SchemaField(
        name="customer", field_type="RECORD", description="Customer snapshot at order time",
        fields=[
            bigquery.SchemaField("name", "STRING"),
            bigquery.SchemaField("email", "STRING"),
            bigquery.SchemaField(
                name="address", field_type="RECORD",
                fields=[
                    bigquery.SchemaField("city", "STRING"),
                    bigquery.SchemaField("country", "STRING"),
                    bigquery.SchemaField("postcode", "STRING"),
                ],
            ),
        ],
    ),
    bigquery.SchemaField("tags", "ARRAY", mode="REPEATED", description="Free-form tags"),
    bigquery.SchemaField("metadata", "JSON", description="Source-specific attributes"),
]

def print_schema(schema, indent=0):
    for field in schema:
        print(f"{'  ' * indent}{field.name:16} {field.field_type:12} {field.mode:10} "
              f"{(field.description or '')[:40]}")
        if field.fields:
            print_schema(field.fields, indent + 1)


print_schema(ORDER_SCHEMA)

## 2. Create the table

Partition by the business date, cluster by the two most-filtered columns.

In [ ]:
table = bigquery.Table(f"{PROJECT_ID}.curated.orders_full", schema=ORDER_SCHEMA)
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="order_date",
    expiration_ms=730 * 24 * 60 * 60 * 1000,
    require_partition_filter=True,
)
table.clustering_fields = ["customer_id", "channel"]
table.description = "Fully modelled orders with nested customer and line items"
table.labels = {"tier": "curated", "owner": "data-eng"}

table = client.create_table(table, exists_ok=True)
print(f"✓ {table.full_table_id}")
print(f"  partition:  {table.time_partitioning.field}")
print(f"  clustering: {table.clustering_fields}")

## 3. Load nested data from JSON

`insert_rows_json` accepts a Python dict directly — nested dicts and lists map
straight onto `RECORD` and `REPEATED` fields.

In [ ]:
orders = [
    {
        "order_id": "o1001",
        "customer_id": "c0001",
        "order_ts": "2026-01-15T09:15:00Z",
        "order_date": "2026-01-15",
        "channel": "web",
        "region": "EMEA",
        "currency": "EUR",
        "amount_local": 65.50,
        "amount_eur": 65.50,
        "status": "complete",
        "line_count": 2,
        "line_items": [
            {"product_id": "p0001", "quantity": 1, "unit_price": 25.00, "discount": 0.0},
            {"product_id": "p0002", "quantity": 2, "unit_price": 20.25, "discount": 0.0},
        ],
        "customer": {
            "name": "Ada Lovelace",
            "email": "ada@example.com",
            "address": {"city": "London", "country": "GB", "postcode": "EC1A"},
        },
        "tags": ["priority", "gift"],
        "metadata": {"campaign": "cmp_007", "device": "desktop"},
    },
    {
        "order_id": "o1002",
        "customer_id": "c0002",
        "order_ts": "2026-01-15T14:42:00Z",
        "order_date": "2026-01-15",
        "channel": "mobile",
        "region": "AMER",
        "currency": "USD",
        "amount_local": 129.99,
        "status": "active",
        "line_count": 1,
        "line_items": [
            {"product_id": "p0003", "quantity": 1, "unit_price": 129.99, "discount": 5.0},
        ],
        "customer": {"name": "Grace Hopper", "email": "grace@example.com"},
        "tags": [],
        "metadata": {"campaign": "cmp_012"},
    },
    {
        "order_id": "o1003",
        "customer_id": "c0003",
        "order_ts": "2026-01-15T18:05:00Z",
        "order_date": "2026-01-15",
        "channel": "store",
        "region": "APAC",
        "currency": "JPY",
        "amount_local": 9800,
        "status": "cancelled",
        "line_count": 3,
        "line_items": [
            {"product_id": "p0004", "quantity": 1, "unit_price": 3000, "discount": 0},
            {"product_id": "p0005", "quantity": 1, "unit_price": 2500, "discount": 0},
            {"product_id": "p0006", "quantity": 1, "unit_price": 4300, "discount": 0},
        ],
        "customer": {"name": "Alan Turing", "email": "alan@example.com"},
        "tags": ["clearance"],
        "metadata": {},
    },
]

errors = client.insert_rows_json(
    client.dataset("curated").table("orders_full"), orders
)
print("Errors:", errors if errors else "none ✅")

In [ ]:
# The JSON functions
df = run_to_dataframe(
    f"""
    SELECT
      order_id,
      customer.name                AS customer_name,
      customer.address.country     AS country,
      JSON_VALUE(metadata, '$.campaign')  AS campaign,
      JSON_VALUE(metadata, '$.device')    AS device,
      ARRAY_LENGTH(tags)            AS tag_count,
      line_count
    FROM `{PROJECT_ID}.curated.orders_full`
    ORDER BY order_id
    """
)
display(df)

In [ ]:
# UNNEST — flattening a repeated field
df = run_to_dataframe(
    f"""
    SELECT
      o.order_id,
      item.product_id,
      item.quantity,
      item.unit_price,
      item.discount,
      ROUND(item.quantity * item.unit_price - COALESCE(item.discount, 0), 2) AS line_total
    FROM `{PROJECT_ID}.curated.orders_full` AS o,
    UNNEST(o.line_items) AS item
    ORDER BY o.order_id, item.product_id
    """
)
display(df)

In [ ]:
# UNNEST an array of scalars
df = run_to_dataframe(
    f"""
    SELECT order_id, tag
    FROM `{PROJECT_ID}.curated.orders_full`,
    UNNEST(tags) AS tag
    ORDER BY order_id, tag
    """
)
display(df)

## 4. Nested aggregation patterns

These are the queries people actually write against nested data.

In [ ]:
# Total revenue from line items, per order
df = run_to_dataframe(
    f"""
    SELECT
      order_id,
      channel,
      SUM(item.quantity * item.unit_price) AS gross_revenue,
      COUNT(*) AS lines,
      ARRAY_AGG(item.product_id ORDER BY item.product_id) AS products
    FROM `{PROJECT_ID}.curated.orders_full` AS o,
    UNNEST(o.line_items) AS item
    GROUP BY order_id, channel
    ORDER BY gross_revenue DESC
    """
)
display(df)

In [ ]:
# Top products by revenue (crossing the order grain)
df = run_to_dataframe(
    f"""
    SELECT
      item.product_id,
      SUM(item.quantity) AS units,
      SUM(item.quantity * item.unit_price) AS revenue,
      COUNT(DISTINCT o.order_id) AS order_count
    FROM `{PROJECT_ID}.curated.orders_full` AS o,
    UNNEST(o.line_items) AS item
    GROUP BY item.product_id
    ORDER BY revenue DESC
    """
)
display(df)

In [ ]:
# Pivot nested data into one row per order
df = run_to_dataframe(
    f"""
    SELECT
      order_id,
      ARRAY_AGG(STRUCT(item.product_id, item.quantity) ORDER BY item.quantity DESC LIMIT 1)[SAFE_OFFSET(0)].product_id AS top_product,
      ARRAY_AGG(STRUCT(item.product_id, item.quantity) ORDER BY item.quantity DESC LIMIT 1)[SAFE_OFFSET(0)].quantity AS top_qty
    FROM `{PROJECT_ID}.curated.orders_full` AS o,
    UNNEST(o.line_items) AS item
    GROUP BY order_id
    """
)
display(df)

## 5. JSON functions reference

| Function | Returns | Use for |
|---------|---------|---------|
| `JSON_VALUE` | A scalar | Extracting a string/number from JSON |
| `JSON_QUERY` | JSON text | Extracting an object/array |
| `JSON_EXTRACT_SCALAR` | A scalar | Legacy name for `JSON_VALUE` |
| `JSON_EXTRACT` | JSON text | Legacy name for `JSON_QUERY` |
| `JSON_QUERY_ARRAY` | An array | Converting JSON to a BigQuery array |
| `PARSE_JSON` | A JSON value | Casting a string to JSON |
| `TO_JSON_STRING` | A string | Converting a value to JSON |

In [ ]:
df = run_to_dataframe(
    f"""
    SELECT
      order_id,
      JSON_VALUE(metadata, '$.campaign')              AS scalar_value,
      JSON_QUERY(metadata, '$.campaign')              AS json_text,
      JSON_EXTRACT_SCALAR(metadata, '$.device')       AS legacy_scalar,
      TO_JSON_STRING(metadata)                        AS whole_object,
      PARSE_JSON(TO_JSON_STRING(metadata))            AS as_json_value
    FROM `{PROJECT_ID}.curated.orders_full`
    ORDER BY order_id
    """
)
display(df)

## 6. Schema evolution

The supported operations, and what they cost.

In [ ]:
# ADD COLUMN — metadata only, fast
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_full` "
    "ADD COLUMN IF NOT EXISTS campaign_id STRING"
)

# DROP a nested field
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_full` "
    "DROP COLUMN IF EXISTS campaign_id"
)

# Relax a REQUIRED field to NULLABLE
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_full` "
    "ALTER COLUMN amount_eur DROP NOT NULL"
)

table = client.get_table(f"{PROJECT_ID}.curated.orders_full")
for field in table.schema:
    print(f"{field.name:16} {field.field_type:12} {field.mode}")

In [ ]:
# Relaxation in the Python client — schema_update_options
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    autodetect=True,
    schema_update_options=[
        bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
        bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION,
    ],
    ignore_unknown_values=True,
    write_disposition="WRITE_APPEND",
)
print("LoadJobConfig with:")
for option in job_config.schema_update_options:
    print(f"  {option}")
print(f"ignore_unknown_values: {job_config.ignore_unknown_values}")

print("\nNote: column type changes (INT64 -> FLOAT64) need ALLOW_FIELD_RELAXATION.")
print("Removing a column from the source NEVER works in BigQuery — only adding and relaxing.")

## 7. Build a JSON schema file and load it

Store schemas in version control. One source of truth, reusable by the CLI and the
Python client.

In [ ]:
def schema_to_json(schema):
    """Convert SchemaField objects into a JSON schema definition."""
    def convert(fields):
        out = []
        for f in fields or []:
            entry = {"name": f.name, "type": f.field_type, "mode": f.mode}
            if f.description:
                entry["description"] = f.description
            if f.fields:
                entry["fields"] = convert(f.fields)
            out.append(entry)
        return out

    return convert(schema)


def schema_from_json(raw):
    """Convert a JSON schema definition into SchemaField objects.

    Recursive: nested RECORD/STRUCT fields are rebuilt to any depth, so a
    three-level schema survives the round trip intact.
    """
    def build(entry):
        children = [build(e) for e in entry.get("fields", [])]
        return bigquery.SchemaField(
            name=entry["name"],
            field_type=entry["type"],
            mode=entry.get("mode", "NULLABLE"),
            description=entry.get("description"),
            fields=children or None,
        )

    return [build(e) for e in raw]


In [ ]:
raw_schema = schema_to_json(ORDER_SCHEMA)
with open("schemas_orders_full.json", "w") as f:
    json.dump(raw_schema, f, indent=2)

round_tripped = schema_from_json(raw_schema)


def depth(fields):
    """Maximum nesting depth of a schema field list."""
    if not fields:
        return 0
    return 1 + max(depth(f.fields) for f in fields)


def field_names(fields):
    """Every field name at every level, in order."""
    names = []
    for f in fields or []:
        names.append(f.name)
        names.extend(field_names(f.fields))
    return names


print("✅ Round trip:")
print(f"   Original:    {len(ORDER_SCHEMA)} top-level fields")
print(f"   Round-trip:  {len(round_tripped)} top-level fields")
print(f"   Names match: {field_names(ORDER_SCHEMA) == field_names(round_tripped)}")
print(f"   Depth:       {depth(ORDER_SCHEMA)} -> {depth(round_tripped)}")

# Every level must survive, not just the top one.
assert field_names(ORDER_SCHEMA) == field_names(round_tripped), "field names lost"
assert depth(ORDER_SCHEMA) == depth(round_tripped), "nesting depth lost"

customer = next(f for f in round_tripped if f.name == "customer")
address = next(f for f in customer.fields if f.name == "address")
print(f"   customer sub-fields:   {len(customer.fields)}")
print(f"   customer.address subs: {len(address.fields)}")
assert len(address.fields) == 3, "level-3 fields were lost"
print("   ✅ Nested structure preserved to full depth")

## 8. Time travel — read the table as it was

The last 7 days are available.

In [ ]:
df = run_to_dataframe(
    f"""
    SELECT COUNT(*) AS n
    FROM `{PROJECT_ID}.curated.orders_full`
      FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
    """
)
display(df)

print("Time travel works on views and materialized views too.")

## 🔑 Takeaways

- Design for the **query patterns**, not the source shape.
- **Never use `FLOAT64` for money** — use `NUMERIC`.
- `RECORD` for fixed nested attributes, `JSON` for sparse evolving ones.
- `UNNEST` is how you cross from order grain to line grain.
- `ARRAY_AGG`, `ARRAY_LENGTH`, `ARRAY_AGG(... ORDER BY ... LIMIT 1)` are your array tools.
- `JSON_VALUE` for scalars, `JSON_QUERY` for objects and arrays.
- BigQuery supports **adding and relaxing** columns — never removing a source column.
- **Store schemas as JSON in version control** and round-trip them.

---

## 📚 Next

[05 · Partitioning](./05-partitioning.ipynb)